# Distancia a la Moneda Ideal y Posicionamiento Global

**Tesis de licenciatura** — Marco Antonio López Lázaro

Este notebook ejecuta el análisis completo desde cero y muestra todos los resultados.

**Repositorio:** https://github.com/marcovanfan101/moneda-ideal-posicionamiento

## Instrucciones

1. Ejecutar las celdas en orden (Shift+Enter o Runtime → Run all)
2. Esperar 3-5 minutos a que termine la descarga de datos
3. Ver los resultados al final del notebook

## 1. Clonar el repositorio

In [ ]:
!git clone https://github.com/marcovanfan101/moneda-ideal-posicionamiento.git

## 2. Entrar a la carpeta del proyecto

In [ ]:
%cd moneda-ideal-posicionamiento

## 3. Instalar dependencias

In [ ]:
!pip install -q yfinance pandas numpy pandas-datareader requests scipy openpyxl matplotlib seaborn

## 4. Ejecutar el pipeline completo

In [ ]:
!python scripts/01_run_all.py

## 5. Cargar y mostrar todos los datos

In [ ]:
import pandas as pd
import numpy as np

# Cargar datos completos
df = pd.read_excel("data/processed/datos_completos.xlsx", index_col=0)

print("=" * 70)
print("TABLA FINAL: DISTANCIA A LA MONEDA IDEAL")
print("=" * 70)

cols = ["Eo", "Sv", "SM2", "D", "Reservas_COFER", "BIS_Turnover"]
print(df[cols].round(4).to_string())

## 6. Índices por divisa (ordenados por D)

In [ ]:
# Ordenar por distancia al ideal
df_sorted = df.dropna(subset=["D"]).sort_values("D")

print("DIVISAS ORDENADAS POR DISTANCIA AL IDEAL (menor = mas ideal)")
print("=" * 70)
print(df_sorted[cols].round(4).to_string())

## 7. Identificación del umbral D ≈ 0.10

In [ ]:
# Divisas cercanas al ideal
cerca = df_sorted[df_sorted["D"] < 0.10]
lejos = df_sorted[df_sorted["D"] >= 0.10]

print(f"Divisas con D < 0.10: {len(cerca)}")
print(f"  Reservas totales: {cerca['Reservas_COFER'].sum():.2f}%")
print(f"  BIS total: {cerca['BIS_Turnover'].sum():.2f}%")
print()
print(f"Divisas con D >= 0.10: {len(lejos)}")
print(f"  Reservas totales: {lejos['Reservas_COFER'].sum():.2f}%")
print(f"  BIS total: {lejos['BIS_Turnover'].sum():.2f}%")

## 8. Correlaciones y regresiones

In [ ]:
# Correlaciones
corr = pd.read_excel("results/tables/correlaciones.xlsx")
print("CORRELACIONES: D vs POSICIONAMIENTO")
print("=" * 70)
print(corr.round(4).to_string(index=False))
print()

# Regresiones
reg = pd.read_excel("results/tables/regresiones.xlsx")
print("REGRESIONES: log(posicionamiento) ~ D")
print("=" * 70)
print(reg.round(4).to_string(index=False))

## 9. Análisis de umbral completo

In [ ]:
umbral = pd.read_excel("results/tables/umbral.xlsx")
print("ANALISIS DE UMBRAL")
print("=" * 70)
print(umbral.round(4).to_string(index=False))

## 10. Robustez: sin Turquía y sin Argentina

In [ ]:
robust = pd.read_excel("results/tables/robustez.xlsx")
print("ANALISIS DE ROBUSTEZ")
print("=" * 70)
print(robust.round(4).to_string(index=False))

## 11. Análisis complementario D2 (sin M2)

In [ ]:
d2 = pd.read_excel("results/tables/d2_sin_m2.xlsx")
print("ANALISIS COMPLEMENTARIO D2 = (100-Eo)*Sv (sin SM2)")
print("=" * 70)
print(d2.round(4).to_string(index=False))

## 12. Visualizaciones

In [ ]:
import matplotlib.pyplot as plt

df_plot = df.dropna(subset=["D", "Reservas_COFER"]).copy()

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Grafico 1: D vs Reservas
ax = axes[0]
data = df_plot[df_plot["Reservas_COFER"] > 0]
ax.scatter(data["D"], data["Reservas_COFER"], c="steelblue", s=100, alpha=0.7)
for idx, row in data.iterrows():
    ax.annotate(idx, (row["D"], row["Reservas_COFER"]), fontsize=9, alpha=0.8)
ax.set_yscale("log")
ax.set_xlabel("D = (100-Eo) * Sv * SM2")
ax.set_ylabel("Reservas COFER (%, escala log)")
ax.set_title("Distancia al Ideal vs Reservas Globales")
ax.grid(True, alpha=0.3)
ax.axvline(x=0.10, color="red", linestyle="--", alpha=0.5, label="Umbral D=0.10")
ax.legend()

# Grafico 2: D vs BIS
ax = axes[1]
data = df_plot[df_plot["BIS_Turnover"] > 0]
ax.scatter(data["D"], data["BIS_Turnover"], c="darkred", s=100, alpha=0.7)
for idx, row in data.iterrows():
    ax.annotate(idx, (row["D"], row["BIS_Turnover"]), fontsize=9, alpha=0.8)
ax.set_yscale("log")
ax.set_xlabel("D = (100-Eo) * Sv * SM2")
ax.set_ylabel("BIS Turnover (%, escala log)")
ax.set_title("Distancia al Ideal vs Turnover Global")
ax.grid(True, alpha=0.3)
ax.axvline(x=0.10, color="red", linestyle="--", alpha=0.5, label="Umbral D=0.10")
ax.legend()

plt.tight_layout()
plt.savefig("results/figures/distancia_vs_posicionamiento.png", dpi=150, bbox_inches="tight")
plt.show()

print("Grafico guardado en: results/figures/distancia_vs_posicionamiento.png")

## 13. Resumen ejecutivo

In [ ]:
print("=" * 70)
print("RESUMEN EJECUTIVO")
print("=" * 70)
print()
print(f"Divisas analizadas: {len(df.dropna(subset=['D']))}")
print(f"Periodo: 2010-2025")
print()
print("HALLAZGO PRINCIPAL:")
print(f"  Umbral en D = 0.10")
print(f"  Divisas con D < 0.10: {len(df_sorted[df_sorted['D'] < 0.10])}")
print(f"  Reservas concentradas: {cerca['Reservas_COFER'].sum():.2f}%")
print()
print("CORRELACIONES:")
print(f"  D vs Reservas: rho = {corr.iloc[0]['Spearman_rho']:.4f} (p = {corr.iloc[0]['Spearman_p']:.4f})")
print(f"  D vs BIS:      rho = {corr.iloc[1]['Spearman_rho']:.4f} (p = {corr.iloc[1]['Spearman_p']:.4f})")
print()
print("CONCLUSION:")
print("  Existe un umbral empirico en D = 0.10 que separa las divisas")
print("  con posicionamiento global de las que no lo tienen.")